In [ ]:
# Notebook: Interactive Inverse Fourier Transform of a Rectangular Spectrum (Sinc Pulse)
# Function to invert: X(j\omega) = A for |\omega| <= T, and 0 otherwise.

import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Define mathematical symbols
t, omega = sp.symbols('t omega', real=True)
A, T_sym = sp.symbols('A T', positive=True, real=True)

# Define the frequency-domain rectangular function X(j\omega)
X_omega = A  # valid for -T <= omega <= T

print("--- 1. SYMBOLIC INTEGRATION (SymPy) ---")
print("Given frequency-domain function X(j\\omega) (Rectangular Spectrum):")
display(sp.Piecewise((A, sp.And(omega >= -T_sym, omega <= T_sym)), (0, True)))

# Calculate the definite integral from -T to T
integrand = X_omega * sp.exp(sp.I * omega * t)
integral_result = sp.integrate(integrand, (omega, -T_sym, T_sym))
x_t_symbolic = (1 / (2 * sp.pi)) * integral_result
x_t_simplified = sp.simplify(x_t_symbolic)

print("\nDerived inverse Fourier transform x(t) (Sinc function):")
display(x_t_simplified)


# ==========================================================
# 2. INTERACTIVE VISUALIZATION (Dynamic Y-limits for Extreme A)
# ==========================================================
def plot_interactive_sinc(T=2.0, A=1.0):
    # Time vector
    t_vals = np.linspace(-10, 10, 1000)
    
    # Compute x(t) with safe handling for t = 0
    with np.errstate(divide='ignore', invalid='ignore'):
        x_vals = (A * T / np.pi) * np.sin(T * t_vals) / (T * t_vals)
        x_vals[t_vals == 0] = A * T / np.pi  # Limit at t = 0 (main lobe peak)
        
    fig, ax = plt.subplots(figsize=(10, 4))
    
    ax.plot(t_vals, x_vals, 'b-', linewidth=2.5, label=f"$x(t) = \\frac{{{A:.1f} \\cdot {T:.1f}}}{{\\pi}} \\text{{sinc}}({T:.1f}t)$")
    
    ax.set_title(rf"Inverse Fourier Transform: Sinc Pulse ($T = {T:.1f}, A = {A:.1f}$)", fontsize=12)
    ax.set_xlabel(r"Time $t$ (s)", fontsize=11)
    ax.set_ylabel(r"$x(t)$", fontsize=11)
    ax.grid(True, linestyle=":", alpha=0.7)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    
    # Dynamic y-axis limits based on the maximum possible peak value (A*T / pi) plus a safety margin
    max_peak = abs(A * T / np.pi)
    ylim_margin = max(max_peak * 1.3, 1.0)
    
    ax.set_xlim(-10, 10)
    ax.set_ylim(-ylim_margin, ylim_margin)
    ax.legend(fontsize=11)
    
    plt.tight_layout()
    plt.show()

# Execute interactive widget with explicit sliders for both T and A
interact(
    plot_interactive_sinc, 
    T=FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1, description="Parameter T", style={'description_width': 'initial'}),
    A=FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.1, description="Amplitude A", style={'description_width': 'initial'})
);